# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tessa-Saumu/FlyRank-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

> **Freestyle submission, V2 edition.** This notebook belongs to my final, freestyle capstone: *which ranking pages are losing clicks to AI-answer search before it shows, and which should an editor protect first?* The pipeline was rebuilt as **V2**: the whole process was written down and hashed in `work/V2_SPEC.md` before any V2 code ran. V1 stays in the repository as the first process, and every V1 number shown here is labelled V1. My earlier Lane 2 (refresh) experiments are kept for the record in `work/archive/` and are not part of this submission.

**What this card asks:** five plain-word contract answers, exactly three verification queries on a mid-panel month, at
most five features each with "knowable at the decision moment because...", and a deliberate label-derived leak shown,
then removed. **The principle it serves:** data hygiene first. One explicit grain, null is not zero, and nothing from
the future in a feature.

In [1]:
# Setup: works locally and in Colab. Reads committed receipts; the warehouse is touched only by the opt-in rerun.
import importlib.util, json, os, sys
if any(importlib.util.find_spec(p) is None for p in ["duckdb", "sklearn", "scipy", "matplotlib"]):
    get_ipython().run_line_magic("pip", "install -q duckdb huggingface_hub pandas scikit-learn scipy matplotlib")
here = os.path.abspath(os.getcwd())
ROOT = next((p for p in [here, os.path.dirname(here), os.path.dirname(os.path.dirname(here))]
             if os.path.isdir(os.path.join(p, "work", "scripts"))), None)
if ROOT is None:  # fresh Colab runtime: fetch the code (no data lives in git)
    get_ipython().system("git clone -q https://github.com/Tessa-Saumu/FlyRank-ML-Internship.git")
    ROOT = os.path.abspath("FlyRank-ML-Internship")
sys.path.insert(0, os.path.join(ROOT, "work", "scripts"))
OUT, FIG = os.path.join(ROOT, "work", "outputs"), os.path.join(ROOT, "work", "figures")
import numpy as np, pandas as pd
import warnings; warnings.filterwarnings("ignore")
from IPython.display import Image, display, Markdown
import wh
RERUN_PIPELINE = False   # opt-in: True recomputes from the pinned snapshot (needs HF_TOKEN). V2 steps then run in a scratch copy, never in the repo.
DATA = wh.data_available(try_download=RERUN_PIPELINE)
print("pinned warehouse snapshot available:", DATA, "| revision", wh.REV[:8], "| RERUN_PIPELINE =", RERUN_PIPELINE)
receipt = lambda name: json.load(open(os.path.join(OUT, name)))
pd.set_option("display.width", 170); pd.set_option("display.max_columns", 30)

# V2 receipts (aggregates only) and the helpers every notebook shares.
v2 = lambda n: receipt(f"v2_{n}.json")
BASE = ["rule_position_slipping", "mean_reversion_ctr_excess", "client_shock", "persistence_prev_ratio"]
BASE_LABEL = {"rule_position_slipping": "editor's rule", "mean_reversion_ctr_excess": "CTR above peers",
              "client_shock": "client shock", "persistence_prev_ratio": "persistence"}
def v2_months():
    """(month, role, evaluation block) for the four evaluated months. The logistic champion is in every block."""
    vf, cf, ts = v2("validation_folds"), v2("confirmation_apr"), v2("test")
    return [("2026-02", "validation", vf["folds"]["validation_1"]), ("2026-03", "validation", vf["folds"]["validation_2"]),
            ("2026-04", "confirmation", cf["evaluation"]), ("2026-05", "test", ts["evaluation"])]
def champ(ev):
    return ev["models"].get("LOGISTIC") or ev["models"]["SELECTED:LOGISTIC"]
def fold_table():
    """One row per evaluated month: the champion against the four frozen baselines, same rows, same metric."""
    rows = []
    for m, role, ev in v2_months():
        lg, best = champ(ev), ev["best_baseline_by_ap"]
        rows.append({"month": m, "role": role, "base rate": ev["base_rate"], "rule P@50": ev["models"][BASE[0]]["p@50"],
                     "best baseline (by AP)": BASE_LABEL[best], "best baseline AP": ev["models"][best]["ap"],
                     "V2 AP": lg["ap"], "V2 AP 95% CI": lg["ap_ci"], "V2 P@50": lg["p@50"], "V2 P@50 95% CI": lg["p@50_ci"],
                     "P@50, unmeasurable = miss": lg["p@50_unmeasurable_as_miss"],
                     "rule P@50, unmeasurable = miss": ev["models"][BASE[0]]["p@50_unmeasurable_as_miss"],
                     "rule share of click shortfall (top 50)": ev["models"][BASE[0]]["clicks_lost_captured"],
                     "V2 share of click shortfall (top 50)": lg["clicks_lost_captured"]})
    return pd.DataFrame(rows)

pinned warehouse snapshot available: True | revision 50cbf7c3 | RERUN_PIPELINE = False


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Question:** one row is one what, over which dates? **Why it matters (FlyRank principle: one explicit grain):** every
count, join and label below depends on it. **Method:** five plain-word answers, then queries to verify them in section 3.

1. **What is one row?** One top-10 page at one monthly decision point (page x origin month), built from the daily
   performance fact table (report_date x client x content).
2. **Which tables?** `fact_content_daily_performance` (Search Console and GA4, daily), `dim_content` (page
   metadata) and `dim_clients` (data start dates). Warehouse build v20260703, pinned revision `50cbf7c3`.
3. **Which window?** Features use the two closed months before the decision. The label uses the two months after it.
   Origins run December 2025 to May 2026, after Google's September 2025 change that removed deep-result impressions,
   plus a live origin in July 2026 that has no outcome yet.
4. **Label or proxy?** Observed: clicks in the outcome window below 75% of the clicks the page's current CTR predicts at
   its new position.
5. **One exclusion.** Pages whose expected outcome clicks are under 10 (4% to 16% per evaluated origin, mostly pages that lost
   visibility) are excluded because their label cannot be measured. This uses outcome-window impressions, so it is a
   disclosed population choice, and ML-09 reports what it hides.

In [2]:
summ = receipt("erosion_frame_summary.json")
display(pd.DataFrame(summ["origins"])[["origin", "eligible", "measurable", "clients", "lost_visibility"]])

,origin,eligible,measurable,clients,lost_visibility
0,2025-12,5905,5378.0,19,0.0892
1,2026-01,6642,5785.0,19,0.1290
2,2026-02,7630,7083.0,21,0.0717
3,2026-03,8984,8613.0,29,0.0413
4,2026-04,12319,11186.0,30,0.0920
5,2026-05,13082,NaN,33,NaN
6,2026-07,14100,NaN,37,NaN


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Question:** which fields do I touch, and in which bucket? **Why it matters (FlyRank principle: nothing from the future
in a feature):** a field in the wrong bucket is how leaks start.

| Bucket | Fields | Note |
|---|---|---|
| **Feature** | the five core features below, plus the 36 features of the V2 model (31 carried from V1, 5 new date and rhythm features) | all computed from months before the decision; list exported to `v2_features.json` |
| **Label** | `erode` (primary); `erode_rel` (secondary, client-relative, never used to choose a model) | observed in the two months after the decision |
| **Context** | `origin` (decision month), `client_hash_id` (grouping and splits only), `content_hash_id` (joins and tie-breaks only) | pseudonymous; never displayed or used as a feature |
| **Excluded** | `clk_out`, `imp_out`, `pos_out`, `exp_out` (outcome-window columns, the label's own ingredients); `provider_used`; the 90-day query table; `model_used`; `last_optimized_date`; and three V1 features, `page_prev_ratio`, `client_prev_erode_rate`, `word_count` | why: see below |

**Why each exclusion.**
- **Outcome-window columns** are the label's ingredients, so as features they would be the answer (the leak demo in
  section 3 shows it).
- **`provider_used`** contains strings that look like client brand names in some rows. It is never selected anywhere in
  this project (privacy rule).
- **The 90-day query table** covers April to June 2026, which overlaps every outcome window.
- **`model_used`** records the latest regeneration of a page, not its author, so it is not a valid treatment.
- **`last_optimized_date`** covers 8.7% of pages, all dated after 24 April 2026, which is too little follow-up time.
- **`page_prev_ratio` and `client_prev_erode_rate`** (V1 features) are empty in the earliest training rows, because no origin
  exists four months back, and the label anti-persists. `page_prev_ratio` survives only as the input of the frozen persistence
  baseline.
- **`word_count`** comes from `dim_content`, a July 2026 snapshot. 86% to 99.9% of eligible pages were updated on or after
  each origin, so the word count may describe a later regeneration, not the page at the decision moment.

**Five core features, each knowable at the decision moment** (the model uses 36, ML-08):

| Feature | Meaning | Knowable at the decision moment because... |
|---|---|---|
| `l_imp_f` | log impressions over months *o-2* and *o-1* | both months close before day 1 of month *o* |
| `pos_f` | impression-weighted average position, same window | same window, computed from `gsc_sum_position / impressions` |
| `ctr_vs_ref` | log of shrunk CTR over the fixed position curve | the curve uses Oct-Nov 2025 only, before every outcome window |
| `ctr_growth` | log CTR change from month *o-2* to month *o-1* | both months are closed |
| `days_since_last_click` | days from the last click day in the window to the origin's first day (new in V2) | read from daily rows inside months *o-2* and *o-1* only; the latest daily date read for each origin is asserted to be before its first day |

In [3]:
feats = v2("features")
names_ = [f["name"] for f in feats["features"]]
five = ["l_imp_f", "pos_f", "ctr_vs_ref", "ctr_growth", "days_since_last_click"]
print(len(names_), "features exported by the V2 code;", feats["n_v1_features"], "in V1,", len(feats["removed"]), "removed,", len(feats["added"]), "added")
print("the five core features are in the exported feature list:", {f: f in names_ for f in five})
print("label ingredients or removed V1 features present in the feature list (must be empty):",
      [c for c in ["imp_out", "clk_out", "pos_out", "exp_out", "erode", "log_ratio_out", "page_prev_ratio", "client_prev_erode_rate", "word_count"] if c in names_])
print("latest daily date read for the rhythm features, per origin (all before the origin's first day):")
display(pd.DataFrame(feats["timeline_check_rhythm"]).T[["max_daily_date_used", "origin_first_day"]])

36 features exported by the V2 code; 34 in V1, 3 removed, 5 added
the five core features are in the exported feature list: {'l_imp_f': True, 'pos_f': True, 'ctr_vs_ref': True, 'ctr_growth': True, 'days_since_last_click': True}
label ingredients or removed V1 features present in the feature list (must be empty): []
latest daily date read for the rhythm features, per origin (all before the origin's first day):


,max_daily_date_used,origin_first_day
2025-12,2025-11-30,2025-12-01
2026-01,2025-12-31,2026-01-01
2026-02,2026-01-31,2026-02-01
2026-03,2026-02-28,2026-03-01
2026-04,2026-03-31,2026-04-01
2026-05,2026-04-30,2026-05-01


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Question:** does the contract hold on the real data? **Why it matters (FlyRank principle: a claim without a query is a
guess):** three checks prove the grain, the size and date span, and the availability handling before anything is built on
them. **Method:** exactly three queries on a mid-panel month, March 2026, away from the final months. With the pinned
snapshot on disk the queries run live and are compared with the committed receipt; without it the cell reads the receipt.

In [4]:
contract = receipt("data_contract.json")
if DATA:
    c = wh.con()
    d = wh.DAILY_DIR + "/month=2026-03/*.parquet"
    q1 = c.sql(f'''SELECT report_date, content_hash_id, COUNT(*) n FROM read_parquet('{d}')
                   GROUP BY 1, 2 HAVING COUNT(*) > 1 LIMIT 5''').df()
    q2 = c.sql(f'''SELECT COUNT(*) rows_, MIN(report_date)::VARCHAR first_day, MAX(report_date)::VARCHAR last_day,
                   COUNT(DISTINCT client_hash_id) clients, COUNT(DISTINCT content_hash_id) pages FROM read_parquet('{d}')''').df()
    q3 = c.sql(f'''SELECT SUM((gsc_data_available IS TRUE)::INT) gsc_true, SUM((gsc_data_available IS NOT TRUE)::INT) gsc_not_true,
                   SUM((gsc_data_available IS NULL)::INT) gsc_null, SUM((ga4_data_available IS TRUE)::INT) ga4_true,
                   SUM((ga4_data_available IS NULL)::INT) ga4_null FROM read_parquet('{d}')''').df()
    live = {"q1_duplicate_grain_rows": len(q1), "q2": json.loads(q2.to_json(orient="records"))[0], "q3": json.loads(q3.to_json(orient="records"))[0]}
    print("queries re-run live on the pinned snapshot; identical to the committed receipt:", all(live[k] == contract[k] for k in live))
    contract.update(live)    # shown below, not written back
print("Q1 grain probe, duplicate (report_date, content) rows:", contract["q1_duplicate_grain_rows"], "(must be 0)")
print("Q2 size and span:", contract["q2"])
print("Q3 availability flags (three-valued, so filter with IS TRUE):", contract["q3"])

queries re-run live on the pinned snapshot; identical to the committed receipt: True
Q1 grain probe, duplicate (report_date, content) rows: 0 (must be 0)
Q2 size and span: {'rows_': 9841378, 'first_day': '2026-03-01', 'last_day': '2026-03-31', 'clients': 55, 'pages': 331437}
Q3 availability flags (three-valued, so filter with IS TRUE): {'gsc_true': 3611061.0, 'gsc_not_true': 6230317.0, 'gsc_null': 0.0, 'ga4_true': 413966.0, 'ga4_null': 3018741.0}


**What it means.** The grain holds (zero duplicates), March has the expected one-month span, and the availability flags
contain NULLs as well as TRUE and FALSE. That is why every query in this project filters with `IS TRUE`, and why GA4
columns are summed only on days where `ga4_data_available IS TRUE` (earlier days are zero-filled, and zero there does
not mean "no engagement").

**A deliberate leak, shown and then removed** (a model check, not a fourth query). **Why:** to prove the evaluation can
see a leak. **Method:** train a logistic model on the five core features with the V2 split of validation fold 2 (train
December and January, score March). Then add the outcome-window clicks as a "feature" (the label's own ingredient) and
score the same month again. The receipt is `v2_summary.json`, written by `work/scripts/v2_summary.py`.

In [5]:
if RERUN_PIPELINE and DATA:
    import v2_summary; v2_summary.main()     # about a minute; rewrites v2_summary.json only
demo = receipt("v2_summary.json")["contract_leak_demo"]
display(pd.DataFrame({k: demo[k] for k in ["five_core_features", "with_leak_clk_out"]}).T)
fold2 = v2("validation_folds")["folds"]["validation_2"]
print(f"base rate in {demo['score_origin']}: {demo['base_rate_2026-03']} | the full 36-feature champion on the same fold: "
      f"AP {fold2['models']['LOGISTIC']['ap']}, P@50 {fold2['models']['LOGISTIC']['p@50']}")

,p@50,pr_auc
five_core_features,0.44,0.375
with_leak_clk_out,1.00,1.000


base rate in 2026-03: 0.338 | the full 36-feature champion on the same fold: AP 0.4767, P@50 0.7


**What it means.** With the outcome's own ingredient as a feature, the ranking becomes perfect (P@50 1.00, AP
1.00). That is the signature of a leak, not skill. With it removed, the five core features alone reach P@50
0.44 and AP 0.375 against a base rate of 0.338, while the
full 36-feature champion reaches AP 0.477 and P@50 0.70 on the same fold. The core idea is real but thin on its
own, which is why the study engineers more features. The leaky column is never used again. Feature timing is
asserted in code: `erosion_frame.py` builds features only from months before the origin, and the V2 rhythm code asserts the latest daily
date it reads (table above).

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Question:** what can this data never tell me? **Why it matters (FlyRank principle: careful words):** a limit stated up
front keeps a claim from outrunning its data.

- **No AI-answer flag.** Search Console does not record whether an AI answer appeared for a query. So this study
  measures click loss at a stable rank, which is **consistent with** AI answers absorbing clicks. Seasonality, changes
  to other search features and impression inflation are competing explanations. ML-06 tests the ones the data allows.
- **An unbalanced panel, and about five effective clients.** Search Console clients grow from 2 to 65 over 17 months, so any
  pooled trend mixes in onboarding. Within an evaluated month the top three clients hold 61% to
  74% of the rows and the effective number of clients is 4.42 to 5.80.
  Trends are estimated within page or within client, and every interval resamples clients.
- **GA4 starts late.** GA4 (and with it AI-referral tracking) starts in October 2025 with 4 clients. Earlier days are
  zero-filled, so zero there does not mean "no engagement".
- **A measurement change.** Google's September 2025 removal of deep-result impressions shifts impressions toward the top
  10, so every window starts in October 2025 or later.
- **A selected population.** Only top-10 pages with enough clicks, and only pages whose outcome is measurable (4.13% to
  16.09% dropped per evaluated month), so the results describe pages that still rank.
- **Snapshot attributes.** `search_volume`, `cpc`, `competition`, `main_intent`, `content_type` and the keyword attributes are as of
  July 2026, not as of each origin. `keyword_created_date` is checked in code to be on or before the origin (zero violations in
  every origin, `v2_features.json`).
- **One warehouse revision, and the data ends on 2026-06-30.** There is no more data to collect a fresh holdout from.

In [6]:
mon = pd.DataFrame(receipt("eda_ai_monthly.json")["months"])
display(mon[["month", "gsc_clients", "ga4_clients"]])
snap = pd.DataFrame(v2("features")["snapshot_diagnostics_dim_content"]).T
display(snap[["eligible_pages", "share_content_updated_on_or_after_origin", "keyword_created_after_origin_count", "keyword_created_missing_share"]])

,month,gsc_clients,ga4_clients
0,2025-01,2,0
1,2025-02,3,0
2,2025-03,4,0
3,2025-04,4,0
4,2025-05,4,0
5,2025-06,9,0
6,2025-07,16,0
7,2025-08,15,0
8,2025-09,23,0
9,2025-10,31,4


,eligible_pages,share_content_updated_on_or_after_origin,keyword_created_after_origin_count,keyword_created_missing_share
2025-12,5905.0,0.9985,0.0,0.0073
2026-01,6642.0,0.9977,0.0,0.0089
2026-02,7630.0,0.9987,0.0,0.0077
2026-03,8984.0,0.8649,0.0,0.0076
2026-04,12319.0,0.8937,0.0,0.0056
2026-05,13082.0,0.9094,0.0,0.0058


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.